# 3.5 匹配特徵越多，為什麼要除以平方根？


同樣兩個候選，softmax 對分數 `[1,0]` 給約 `[0.731,0.269]`，對 `[8,0]` 卻給約 `[0.9997,0.0003]`。分數差越大，讀法越集中。若只增加匹配特徵數，就順帶讓分數起伏大很多，我們可能無意間改變讀取方式。

點積把每格乘積相加。先想像各格乘積的取值互不影響，這叫獨立；每格有一半機會為 -1、一半為 +1；它的平均是 0，平方偏離平均後再取平均為 1，這個起伏量叫**變異數**。取變異數平方根得到**標準差**，能用原數字的尺度看典型起伏。

加兩格時，四種等可能結果是 -2、0、0、2。變異數為 `(4+0+0+4)/4=2`，標準差為 √2，沒有直接翻倍，因為正負可能抵消。對 D 個獨立、平均為 0、變異數為 1 的格子，加總變異數為 D，標準差為 √D。

因此常將 Q/K 點積除以 `sqrt(D)`，D 是每個匹配向量的格數。這叫**縮放點積注意力**，完整讀取為 `softmax(QKᵀ/√D)V`。四格除以 2，六十四格除以 8，使指定假設下的典型起伏接近同一尺度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math
import torch

torch.manual_seed(42)
for d in [4, 64]:
    q = torch.randn(1000, d)
    k = torch.randn(1000, d)
    scores = (q * k).sum(dim=-1)
    scaled = scores / math.sqrt(d)
    print(d, scores.std().item(), scaled.std().item())

`randn` 產生平均約 0、標準差約 1 的亂數，每組 q、k 有 d 格，共一千組。`q*k` 逐格相乘，`sum` 得每組點積，`std()` 量這一千個結果的起伏。

d=4 時，未縮放標準差應靠近 2；d=64 時靠近 8。縮放後兩者都靠近 1，樣本有限所以不會恰好等於理論值。這次是檢查尺度推算，沒有做完整匹配表或訓練。

真實訓練後的 q/k 未必符合獨立、零平均、每格變異數為 1 的假設，所以除以 √D 不保證分數永遠同樣大。練習先讓 q、k 都乘 2，點積各乘四倍，縮放前後標準差也都約四倍。它控制特徵數的影響，不能抵消全部數值變化。

<details>
<summary>補充與重做</summary>

「獨立」表示知道一格取什麼，不改另一格的機會。變異數相加用到這項假設。√D 的效果與 [1.15](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html) 的分數除溫度相似，但這裡控制的是注意力匹配，生成溫度控制的是選下一字。分成多頭時，D 要用每頭 Q/K 的寬度。

</details>
